# CIEDep-Net — end-to-end walkthrough

이 노트북은 `src/ciedep` 모듈만 사용해 논문 파이프라인을 한 번에 훑습니다.
스크립트(`scripts/01_*.py` ~ `08_*.py`)와 같은 코드를 호출하므로,
여기서 확인한 뒤 대규모 실행은 스크립트로 돌리면 됩니다.

1. 설정 / 라벨
2. 데이터셋 구축 (참가자 발화 추출)
3. 턴 분리 → Q-A 대화 구조
4. 특징 추출 (멜, Wav2Vec 2.0)
5. LLM 해석 (CoT + self-consistency)
6. 모델 구성과 forward
7. 학습 / 평가
8. 시각화

## 0. 설정

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from ciedep.config import load_config

CONFIG_PATH = ROOT / "configs" / "daic_woz.yaml"     # or e_daic.yaml
cfg = load_config(CONFIG_PATH)

dataset_cfg, paths, audio_cfg = cfg["dataset"], cfg["paths"], cfg["audio"]
print(f"dataset      : {dataset_cfg['name']}")
print(f"window / hop : {audio_cfg['window_sec']}s / {audio_cfg['hop_sec']}s")
print(f"pad length   : {audio_cfg['pad_length']}")
print(f"LLM          : {cfg['llm']['model_id']} ({cfg['llm']['prompt_strategy']})")

## 1. 라벨

PHQ-8 을 3구간(0-9 / 10-14 / 15-24)으로 나눠 층화 분할과 증강 대상 선정에 쓴다.

In [ ]:
from ciedep.data import build_label_lookup, load_labels

label_df = load_labels(
    dataset_cfg["label_csv"],
    dataset_cfg["label_columns"],
    dataset_cfg.get("excluded_indices", ()),
)
label_lookup = build_label_lookup(label_df)

print(f"참가자 {len(label_df)}명")
print(label_df["phq_class"].value_counts().sort_index())
label_df.head()

## 2. 데이터셋 구축

전사의 시간 정보로 참가자 발화 구간만 잘라 이어 붙인다.
면담자 음성은 우울증 분석 편향을 막기 위해 전부 제거한다.

In [ ]:
from ciedep.data import extract_participant_audio

raw_dir = Path(dataset_cfg["raw_dir"])
sample_dir = sorted(p for p in raw_dir.iterdir() if p.is_dir())[0]

audio = extract_participant_audio(
    sample_dir,
    dataset_cfg["transcript_columns"],
    participant_label=dataset_cfg["participant_label"],
    sample_rate=audio_cfg["sample_rate"],
    audio_suffix=dataset_cfg["audio_suffix"],
    transcript_suffix=dataset_cfg["transcript_suffix"],
)
print(f"{sample_dir.name}: {audio.shape[0] / audio_cfg['sample_rate']:.1f}s (참가자 발화만)")

# 전체 참가자 처리는 아래 한 줄 (scripts/01_build_dataset.py 와 동일)
# from ciedep.data import build_participant_dataset
# build_participant_dataset(raw_dir, paths["participant_dir"], dataset_cfg["transcript_columns"], ...)

## 3. 턴 분리 → Q-A 대화 구조

한 화자의 발화 시작부터 상대 화자의 발화 시작 직전까지를 한 턴으로 정의하고,
턴 안의 발화 구간들을 이어 붙인 뒤 화자 라벨을 붙여 대화 구조를 만든다.

In [ ]:
import pandas as pd
from ciedep.data import extract_turn_pairs
from ciedep.llm import dialogue_from_reference_transcript

transcript_df = pd.read_csv(next(sample_dir.glob(f"*{dataset_cfg['transcript_suffix']}")))

turns = extract_turn_pairs(
    transcript_df,
    dataset_cfg["transcript_columns"],
    dataset_cfg["interviewer_label"],
    dataset_cfg["participant_label"],
)
print(f"턴 {len(turns)}개")

# Whisper 없이 배포본 전사로 구조만 확인 (실제 파이프라인은 턴 오디오를 Whisper-large 로 전사)
dialogue = dialogue_from_reference_transcript(
    transcript_df,
    dataset_cfg["transcript_columns"],
    dataset_cfg["interviewer_label"],
    dataset_cfg["participant_label"],
)
print()
print("\n".join(dialogue.splitlines()[:6]))

## 4. 특징 추출

- **인지**: 멜 스펙트로그램 80-bin, 세그먼트별 시간축 평균, 참가자 단위 min-max 정규화
- **표현**: Wav2Vec 2.0 `last_hidden_state` 의 프레임 평균 (768차원)

둘 다 4초 윈도우 / 1초 중첩으로 세그먼트를 만든 뒤 계산한다.

In [ ]:
from ciedep.features import build_mel_transform, extract_mel_features

mel_transform = build_mel_transform(
    sample_rate=audio_cfg["sample_rate"],
    n_mels=audio_cfg["n_mels"],
    frame_length_sec=audio_cfg["frame_length_sec"],
    frame_stride_sec=audio_cfg["frame_stride_sec"],
)

participant_dir = Path(paths["participant_dir"])
wav_path = sorted(participant_dir.glob("*.wav"))[0]

mel = extract_mel_features(
    wav_path, mel_transform,
    sample_rate=audio_cfg["sample_rate"],
    window_sec=audio_cfg["window_sec"],
    hop_sec=audio_cfg["hop_sec"],
    top_db=audio_cfg["top_db"],
)
print(f"{wav_path.stem}: mel {mel.shape}  range=[{mel.min():.3f}, {mel.max():.3f}]")

In [ ]:
# Wav2Vec 2.0 표현 특징 (GPU 권장 — 전체 추출은 scripts/05_extract_features.py)
from ciedep.features import SSLFeatureExtractor

extractor = SSLFeatureExtractor("wav2vec2", cfg["features"]["ssl_model_ids"]["wav2vec2"])
expr = extractor.extract_file(
    wav_path,
    sample_rate=audio_cfg["sample_rate"],
    window_sec=audio_cfg["window_sec"],
    hop_sec=audio_cfg["hop_sec"],
)
print(f"{wav_path.stem}: wav2vec2 {expr.shape}")

## 5. LLM 해석

CoT 프롬프트는 (1) 전체 전사 읽기 → (2) 정서적으로 두드러진 세 순간 선택 →
(3) 인용·근거·해석 정리 → (4) 점수/요약 생성의 네 단계를 지시한다.
Self-consistency 는 같은 프롬프트로 5개를 생성해
점수는 평균, 요약은 평균 임베딩에 가장 가까운 것을 고른다.

In [ ]:
from ciedep.llm import build_prompt, parse_depression_score, parse_inner_summary

messages = build_prompt(dialogue, strategy="cot_self_consistency", target="score")
for m in messages:
    print(f"--- {m['role']} ---")
    print(m["content"][:400])
    print()

In [ ]:
# 파싱 동작 확인
print(parse_depression_score(">>> Depression score: 0.42"))
print(parse_depression_score("I cannot provide that."))       # -> None (재시도 대상)
print(parse_inner_summary(">>> Inner Monologue: I feel drained most days."))

In [ ]:
# 실제 생성 (GPU 필요). 전체 참가자는 scripts/04_llm_interpretation.py 로 실행한다.
# import os; os.environ["HF_TOKEN"] = "..."   # 게이트된 모델일 때만
#
# from ciedep.llm import LLMRunner, run_self_consistency_score, run_self_consistency_summary
# from ciedep.features.text_embed import embed_summaries
#
# runner = LLMRunner(cfg["llm"]["model_id"])
# score, samples = run_self_consistency_score(
#     runner, messages,
#     n_samples=cfg["llm"]["n_samples"],
#     temperature=cfg["llm"]["temperature"],
#     top_p=cfg["llm"]["top_p"],
#     max_new_tokens=cfg["llm"]["max_new_tokens_score"],
# )
# print(score, samples)

## 6. 모델

`build_model` 은 설정과 ablation 플래그로 모델을 만든다.
기본값이 논문의 제안 모델이다.

In [ ]:
import torch
from ciedep.models import AblationFlags, build_model

model_cfg = dict(cfg["model"])
model = build_model(model_cfg, AblationFlags())
print(f"파라미터 {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

B, T = 2, audio_cfg["pad_length"]
out, attn_main, attn_aux = model(
    torch.randn(B, T, model_cfg["d_input"]),      # mel
    torch.rand(B, 1),                             # LLM depression score
    torch.randn(B, model_cfg["d_summary"]),       # inner summary embedding
    torch.randn(B, T, model_cfg["d_expr"]),       # wav2vec2
    return_attn=True,
)
print(f"prediction {tuple(out.shape)} | CA-Fusion attention {tuple(attn_main.shape)}")

In [ ]:
# Ablation 구성별 파라미터 수 (Table III / IV)
variants = {
    "proposed":             AblationFlags(),
    "w/o score":            AblationFlags(use_score=False),
    "w/o summary":          AblationFlags(use_summary=False),
    "w/o cognition":        AblationFlags(use_cognition=False),
    "w/o interpretation":   AblationFlags(use_interpretation=False),
    "w/o expression":       AblationFlags(use_expression=False),
    "baseline 3 (concat)":  AblationFlags(fusion="concat"),
    "baseline 4 (transf.)": AblationFlags(expression_backbone="transformer"),
}
for name, flags in variants.items():
    m = build_model(model_cfg, flags)
    print(f"{name:22s} {sum(p.numel() for p in m.parameters()) / 1e6:6.2f}M")

## 7. 학습 / 평가

`load_bundle` 이 특징과 LLM 산출물을 샘플 순서에 맞춰 정렬한다.
(먼저 4~5단계 스크립트로 특징 파일을 만들어 두어야 한다.)

In [ ]:
from ciedep.pipeline import load_bundle, shuffle_bundle
from ciedep.data import train_test_indices, stratified_folds
from ciedep.utils import set_seed

set_seed(cfg["train"]["seed"])
bundle = shuffle_bundle(load_bundle(cfg), seed=cfg["train"]["seed"])

train_val_idx, test_idx = train_test_indices(
    bundle.labels, test_size=cfg["train"]["test_size"], seed=cfg["train"]["seed"]
)
print(f"전체 {len(bundle)} | train+val {len(train_val_idx)} | test {len(test_idx)}")

In [ ]:
from ciedep.train import train_fold, evaluate_checkpoint
from ciedep.metrics import format_metrics

folds = list(stratified_folds(train_val_idx, bundle.labels,
                              n_splits=cfg["train"]["n_splits"], seed=cfg["train"]["seed"]))

fold, tr_idx, va_idx = folds[0]
record = train_fold(cfg, bundle, fold, tr_idx, va_idx, AblationFlags(), verbose=True)

metrics = evaluate_checkpoint(cfg, bundle, record["checkpoint"], test_idx,
                              AblationFlags(), scaler_reference=tr_idx)
print(format_metrics(metrics))

In [ ]:
# 5-fold 전체는 한 줄로 (scripts/06_train.py 와 동일)
# from ciedep.train import run_cross_validation
# result = run_cross_validation(cfg, ablation=AblationFlags(), tag="proposed")

## 8. 시각화

In [ ]:
from ciedep.analysis import plot_confusion_matrix, plot_prediction_scatter, plot_score_distribution
from ciedep.analysis.plots import plot_loss_curve

plot_loss_curve(record["history"], fold=fold + 1)

In [ ]:
# 학습 곡선 외 결과 그림은 scripts/07_evaluate.py --plots 로 한 번에 생성된다.
# plot_prediction_scatter(y_true, y_pred)
# plot_score_distribution(y_true, y_pred)
# plot_confusion_matrix(y_true, y_pred, class_bins=cfg["augment"]["class_bins"])

## 9. LLM 산출물 타당성

생성 점수를 PHQ-8 범위(x24)로 환산해 실제 점수와 비교하고,
내면 요약은 참가자 발화를 참조문으로 BERTScore 를 계산한다.
논문 보고값: r = 0.69 (p < 0.01), BERTScore = 0.8

In [ ]:
from ciedep.analysis import evaluate_generated_scores
from ciedep.pipeline import llm_filename
from ciedep.utils import load_pickle

scores = load_pickle(Path(paths["llm_dir"]) / llm_filename(cfg["llm"]["model_id"], "score"))
pids = [p for p, s in scores.items() if s is not None and p in label_lookup]

metrics = evaluate_generated_scores(
    [scores[p] for p in pids],
    [label_lookup[p]["phq_score"] for p in pids],
    scale=cfg["llm"]["score_scale"],
)
for key in ("r", "p", "CCC", "MAE", "RMSE"):
    print(f"{key:5s}: {metrics[key]:.4f}")